## What is LangGraph?

LangGraph is a framework to build AI agents using a graph-based workflow.

- You create nodes → each node does some work (Python function, tool, LLM call, memory update, etc.)

- You create edges → these decide what happens next after a node finishes.

- Together, nodes + edges = a graph, and this graph defines the entire agent logic.

So LangGraph = flowchart + AI reasoning + tool execution.

It gives a simple way to build reliable, multi-step, stateful AI agents.

## Core Concepts of LangGraph

1. `Node`

- A node is a function or operation.

- Examples:
    - A function to call an LLM
    - A function to search Google
    - A function to fetch data from an API
    - A function that cleans text

- So Nodes = steps.

2. `Edge`

- An edge decides which node runs next.

    - Can be fixed (Node A → Node B)
    - Can be conditional (If answer is correct → Node X, else → Node Y)

- So Edges = control flow.

3. `State`

- State means data that moves between nodes.

- LangGraph updates this state every time a node runs.

4. `Graph`

- A collection of nodes + edges.

- This graph becomes our agent.

## LangGraph Architecture (Simple View)

`Input` → ``Node 1`` → ``Node 2`` → ``Node 3`` → `Output`

## Basic Steps to Create a Graph

- Step 1: Define the State
- Step 2: Create Nodes
- Step 3: Build the Graph
- Step 4: Run the Graph

## When To Use LangGraph?

Use LangGraph when you need:

- Multi-step agent workflows

    (Example: Research → Summarize → Store in DB → Notify user)

- Strong flow control

    (Example: If LLM confidence < 0.5 → revisit step1)

- Tool calling in sequence

    (Example: search → extract → generate → send response)

- AI apps requiring memory

    (Example: chatbots with persistent session state)

- Agents requiring loops

    (Example: keep trying until success)

## Basic Example of langraph usage:

In [1]:
# step 1 : Define State TypedDict
from typing import TypedDict

class State(TypedDict):
    message: str

In [2]:
# step 2 : Define Nodes

def step1(state: State):
    state["message"] += " Step1 done."
    return state

def step2(state: State):
    state["message"] += " Step2 done."
    return state

In [3]:
# step 3 : Build Graph

from langgraph.graph import StateGraph

graph = StateGraph(State)
graph.add_node("first", step1)
graph.add_node("second", step2)

graph.add_edge("first", "second")
graph.set_entry_point("first")

app = graph.compile()

In [4]:
# step 4 : Invoke Graph
result = app.invoke({"message": "Start."})
print(result)

{'message': 'Start. Step1 done. Step2 done.'}


## Another example 

In [5]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

In [6]:
class MyState(TypedDict):
    value: int
    message: str

In [7]:
def add_ten(state: MyState):
    state["value"] += 10
    state["message"] = f"Added 10. New value = {state['value']}"
    return state

def multiply_two(state: MyState):
    state["value"] *= 2
    state["message"] = f"Multiplied by 2. New value = {state['value']}"
    return state

def subtract_five(state: MyState):
    state["value"] -= 5
    state["message"] = f"Subtracted 5. New value = {state['value']}"
    return state

In [8]:
graph = StateGraph(MyState)
graph.add_node("add_ten", add_ten)
graph.add_node("multiply_two", multiply_two)
graph.add_node("subtract_five", subtract_five)

graph.set_entry_point("add_ten")   

graph.add_edge("add_ten", "multiply_two")
graph.add_edge("multiply_two", "subtract_five")
graph.add_edge("subtract_five", END)

graph_build = graph.compile()

In [9]:
intial_state = {"value": 100, "message": "Starting graph execution."}

final_state = graph_build.invoke(intial_state)
print(final_state)

{'value': 215, 'message': 'Subtracted 5. New value = 215'}
